In [14]:
import numpy as np
import xarray as xr

from eofs.xarray import Eof

In [15]:
import os

import glob

In [3]:
from utils.xr_operators import (
    remove_doy_climatology,
)

In [ ]:
# Read geopotential height data using the xarray module. The file contains
# December-February averages of geopotential height at 500 hPa for the
# European/Atlantic domain (80W-40E, 20-90N).
filename = example_data_path('hgt_djf.nc')
z_djf = xr.open_dataset(filename)['z']

# Compute anomalies by removing the time-mean.
z_djf = z_djf - z_djf.mean(dim='time')

# Create an EOF solver to do the EOF analysis. Square-root of cosine of
# latitude weights are applied before the computation of EOFs.
coslat = np.cos(np.deg2rad(z_djf.coords['latitude'].values)).clip(0., 1.)
wgts = np.sqrt(coslat)[..., np.newaxis]
solver = Eof(z_djf, weights=wgts)

# Retrieve the leading EOF, expressed as the covariance between the leading PC
# time series and the input SLP anomalies at each grid point.
eof1 = solver.eofsAsCovariance(neofs=1)


In [21]:
from pathlib import Path
import numpy as np
import xarray as xr
from eofs.xarray import Eof

def eof_analysis_eof1(
    pwtanom: xr.DataArray,
    *,
    lv: int,
    mon: int,
    neofs: int = 2,
    lat_name: str | None = None,
    lon_name: str | None = None,
    sign_lat: float = -60.0,
    sign_lon: float = 90.0,
    out_dir: str | Path = ".",
    save: bool = True,
):
    """
    NCL-equivalent EOF analysis on weighted anomalies (pwtanom):
      - EOFs over (lat, lon), PCs over time (unit variance)
      - sign convention: ensure EOF1(lat≈-60, lon≈90E) > 0
      - regression pattern = <pwtanom * PC1_std> over time
      - save EOF1 to ERA5.EOF1.z{lv}.Mon{mm}.nc

    Parameters
    ----------
    pwtanom : xr.DataArray
        Weighted anomalies with dims (time, lat, lon). Already demeaned across years for a month.
    lv, mon : int
        Pressure level (for filename) and calendar month.
    neofs : int
        Number of EOF modes to compute (default 2).
    lat_name, lon_name : str or None
        Coordinate names; auto-detected if None.
    sign_lat, sign_lon : float
        Reference point for sign convention (nearest grid point is used).
    out_dir : path-like
        Where to write output NetCDF.
    save : bool
        Write EOF1 file.

    Returns
    -------
    eof1 : xr.DataArray       (lat, lon)
    pc1_std : xr.DataArray    (time,)
    reg : xr.DataArray        (lat, lon) regression pattern
    varfrac1 : float          variance fraction of EOF1 (0..1)
    """
    # --- coord names ---
    if lat_name is None:
        lat_name = "lat" if "lat" in pwtanom.dims else "latitude"
    if lon_name is None:
        lon_name = "lon" if "lon" in pwtanom.dims else "longitude"

    # ensure order (time, lat, lon) and sample dim named 'time'
    da = pwtanom.transpose("time", lat_name, lon_name)

    coslat = np.cos(np.deg2rad(da.coords['lat'].values)).clip(0., 1.)
    wgts = np.sqrt(coslat)[..., np.newaxis]

    # EOF solver: data are already anomalies → center=False (covariance EOFs)
    solver = Eof(da, weights=wgts)

    # EOFs (mode, lat, lon), PCs (time, mode, unit variance)
    eofs = solver.eofs(neofs=neofs, eofscaling=0)

    # eof1 = solver.eofsAsCovariance(neofs=1)[0]
    # eof1 = solver.eofs(neofs=1)[0]
    pcs  = solver.pcs(npcs=neofs, pcscaling=1)
    varf = solver.varianceFraction()  # (mode,)

    print(varf)

    eof1 = eofs.isel(mode=0)
    pc1_std = pcs.isel(mode=0)

    # reference lat/lon (same as NCL)
    sign_lat = -60.0
    sign_lon = 90.0
    
    # find nearest grid point
    lat_vals = eof1['lat'].values
    lon_vals = eof1['lon'].values
    ilat = int(np.argmin(np.abs(lat_vals - sign_lat)))
    
    # normalize lon range for matching (in case of 0–360)
    lon360 = (lon_vals % 360 + 360) % 360
    sign_lon360 = sign_lon % 360
    ilon = int(np.argmin(np.abs(lon360 - sign_lon360)))
    
    # check the EOF sign at that point
    flip = -1.0 if float(eof1.values[ilat, ilon]) < 0 else 1.0
    
    # flip EOF and PC if needed
    eof1 = eof1 * flip
    pc1_std = pc1_std * flip

    # --- save EOF1 like NCL ---
    if save:
        base = Path(out_dir)
        zeof_dir = base / "zeof"
        zeof_dir.mkdir(parents=True, exist_ok=True)
    
        # label level: input level is Pa, convert to hPa
        lv_label = int(lv/100) # hPa
    
        out_path = zeof_dir / f"ERA5.EOF1.z{lv_label}.Mon{mon:02d}.nc"
        eof1.rename("eof1").to_dataset().to_netcdf(out_path)
    return eof1


In [17]:
from pathlib import Path

cwd = Path.cwd()
era5_dir = cwd.parents[6]

# build a path relative to two-up:
gh_dir = era5_dir / 'shared_data/Datasets/ERA5/plev'

1. compute monthly zeof

In [18]:
ilev = 1000 # Pa

def pre(ds): return ds[['var129']].sel(lat=slice(-20,-90)).sel(plev=ilev)

In [19]:
gh_fpath = str(gh_dir / f"era5_an_geopot_reg2_6h_*.nc")

gh_6h_mon = xr.open_mfdataset(gh_fpath, preprocess=pre, combine="by_coords", chunks="auto")
gh_6h_mon = gh_6h_mon.sel(time=slice('1979-01-01', '2023-12-31'))  # years 1979–2023

In [22]:
for mon in np.arange(6,13,1):
    print(mon)
    # pick the month then average within each year -> (year, lat, lon)
    z_mon = (
        gh_6h_mon['var129']
        .where(gh_6h_mon['time'].dt.month == mon, drop=True)
        .groupby('time.year').mean('time')
    )
    
    # convert z (m^2/s^2) -> meters and apply weights AFTER the averaging
    # w = np.sqrt(np.cos(np.deg2rad(z_mon['lat'])))  # √cos(lat), 1D
    pwt_mon = (z_mon / 9.8)                  # weighted monthly fields
    
    # anomalies across years for this month
    clim = pwt_mon.mean('year')
    pwt_anom = pwt_mon - clim
    
    # EOF input: (time, lat, lon)
    da = pwt_anom.rename(year='time').transpose('time', 'lat', 'lon').astype('float32').compute()
    
    eof1 = eof_analysis_eof1(da, lv=ilev, mon=mon)

6
<xarray.DataArray 'variance_fractions' (mode: 45)> Size: 180B
array([4.6238095e-01, 1.8391317e-01, 1.0710790e-01, 6.8594061e-02,
       5.0286926e-02, 4.2113397e-02, 3.0854192e-02, 1.5938032e-02,
       1.0460895e-02, 8.5740434e-03, 5.9711509e-03, 3.4977745e-03,
       2.6842158e-03, 1.2940977e-03, 1.0734544e-03, 8.5545925e-04,
       7.6920172e-04, 6.9221610e-04, 5.4225646e-04, 4.1678184e-04,
       4.0056880e-04, 2.8560014e-04, 2.1786072e-04, 1.9795180e-04,
       1.4750208e-04, 1.2257358e-04, 8.9626359e-05, 7.6001576e-05,
       7.2226714e-05, 5.4690452e-05, 5.3274714e-05, 4.8569244e-05,
       3.6873553e-05, 2.9265790e-05, 2.6435726e-05, 2.2319922e-05,
       1.9777224e-05, 1.9400186e-05, 1.6913462e-05, 1.2600726e-05,
       1.1392456e-05, 7.3937931e-06, 5.8928845e-06, 5.0725475e-06,
       3.3606998e-16], dtype=float32)
Coordinates:
  * mode     (mode) int64 360B 0 1 2 3 4 5 6 7 8 ... 36 37 38 39 40 41 42 43 44
Attributes:
    long_name:  variance_fractions
7
<xarray.DataArray '